<a href="https://colab.research.google.com/github/giyoung00119-source/speech_ASR/blob/main/youtube_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Necessary Libraries

First, we need to install the libraries that will allow us to:
1.  **Extract audio** from YouTube videos (`yt-dlp`).
2.  **Transcribe audio** to text, supporting multiple languages (`openai-whisper`).
3.  **Interact with Google Sheets** to save the results (`gspread`, `google-auth`).

In [ ]:
!pip install -q yt-dlp openai-whisper gspread google-auth
!pip install -q --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

## Authenticate with Google

To access and write to your Google Sheet, we need to authenticate with your Google account. This cell will prompt you to grant access.

In [ ]:
from google.colab import auth
import gspread
import google.auth

auth.authenticate_user()

# Get default credentials after user authentication
credentials, project_id = google.auth.default()

# Authorize gspread client using the obtained credentials
gc = gspread.Client(auth=credentials)

print('Authentication successful!')

Authentication successful!


## Define YouTube URL and Google Sheet ID

Please provide the YouTube URL and the Google Sheet ID where you want to save the transcription.

**Note**: The Google Sheet ID is the long string of characters in the URL between `/d/` and `/edit` (e.g., `1yBRZiyArP5VZ0m-osJSdKv8hrM40eV1fw-i9QAyxQ3w`).

In [ ]:
YOUTUBE_URL = 'https://youtu.be/ft7Bu28iC6Y?si=Lxvye8d_1MxX1QG5' # Replace with your YouTube video URL
GOOGLE_SHEET_ID = '1yBRZiyArP5VZ0m-osJSdKv8hrM40eV1fw-i9QAyxQ3w' # This is the sheet ID from your prompt

## Extract Audio from YouTube Video

Now, we'll use `yt-dlp` to download only the audio stream from the provided YouTube URL. The audio will be saved as an MP3 file.

In [ ]:
import yt_dlp
import os

# audio_filename 변수를 초기화하고 yt-dlp가 실제 파일명을 생성하도록 합니다.
# yt-dlp는 YOUTUBE_URL의 ID를 사용하여 파일을 명명할 것입니다.
ydl_opts = {
    'format': 'bestaudio/best',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '192',
    }],
    'outtmpl': '%(id)s.%(ext)s', # YouTube 비디오 ID를 파일명으로 사용
    'quiet': True, # yt-dlp의 출력을 억제합니다.
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info_dict = ydl.extract_info(YOUTUBE_URL, download=True)
    # yt-dlp가 실제로 사용한 파일명을 가져옵니다.
    # 일반적으로 단일 파일 다운로드의 경우 첫 번째 요청된 다운로드의 파일 경로가 됩니다.
    audio_filename = os.path.join(os.getcwd(), info_dict['id'] + '.mp3')
    print(f"Audio extracted and saved as {audio_filename}")

Audio extracted and saved as /content/ft7Bu28iC6Y.mp3


## Transcribe Audio using Whisper

Next, we'll load the Whisper ASR model and use it to transcribe the extracted audio. The `small` model is a good balance of speed and accuracy for general purposes and supports multiple languages. We will also ensure we get timestamps for each segment.

In [ ]:
import whisper

print("Loading Whisper model... This might take a moment.")
model = whisper.load_model("small") # You can choose 'tiny', 'base', 'small', 'medium', 'large'

print("Transcribing audio...")
result = model.transcribe(audio_filename, fp16=False) # fp16=False for CPU or older GPUs

print("Transcription complete!")
# print(result["text"])


Loading Whisper model... This might take a moment.
Transcribing audio...
Transcription complete!


## Save Transcription to Google Sheet

Finally, we'll take the transcribed text along with its start and end timestamps and write it to the specified Google Sheet. Each segment will occupy a row with columns for 'Start Time', 'End Time', and 'Transcription'.

In [ ]:
import pandas as pd

sh = gc.open_by_key(GOOGLE_SHEET_ID)
worksheet = sh.get_worksheet(0) # Get the first worksheet

# Prepare data for the DataFrame
data_to_sheet = []
for segment in result['segments']:
    start_time = segment['start']
    end_time = segment['end']
    text = segment['text'].strip()
    data_to_sheet.append({'Start Time': start_time, 'End Time': end_time, 'Transcription': text})

df_transcription = pd.DataFrame(data_to_sheet)

# Clear existing content and write headers and data
worksheet.clear()
worksheet.update([df_transcription.columns.values.tolist()] + df_transcription.values.tolist())

print(f"Transcription saved to Google Sheet: {sh.url}")

Transcription saved to Google Sheet: https://docs.google.com/spreadsheets/d/1yBRZiyArP5VZ0m-osJSdKv8hrM40eV1fw-i9QAyxQ3w
